# ST-GAE: Real-Time Spatiotemporal Anomaly Detection for Toronto Mobility

This notebook implements a **weakly supervised Spatiotemporal Graph Autoencoder (ST-GAE)** for continuous congestion / collision risk scoring on Toronto Bluetooth monitored corridors.

## Approach

Direct collision prediction is unreliable: crashes are rare, sparse, and poorly balanced as labels. Instead we:

1. Learn the **physics of healthy traffic** under ordinary weather and calendar conditions.
2. Score every corridor × 5-minute step by **reconstruction error** $S_{i,t} = \|X_{i,t} - \hat{X}_{i,t}\|^2$.
3. Treat documented collisions as an **evaluation resource** (thresholds, lead-time, PR-AUC), not as scarce training targets.


## Steps

| Step | Section | Purpose |
|---|---|---|
| 1 | Setup, load, clean, normative mask | Reproducible baseline data for training |
| 2 | Feature tensor + corridor graph | Build $X \in \mathbb{R}^{T \times N \times D}$ and adjacency $A$ |
| 3 | ST-GAE train / persist | GCN + GRU autoencoder on normal windows only |
| 4 | Realtime-style scoring | Streaming windows, route thresholds, collision validation |

**Data prerequisite:** upload `route_time_panel_v1.parquet` to Google Drive at `MyDrive/Dissertation/` (built locally with `python -m src.processing --step panel_v1`).

## 0. Colab dependency install

Install the Data Processing + ML stack used by later cells (`pandas`, `numpy`, `sklearn`, `pyarrow`, `matplotlib` `torch`, `torch_geometric`, plus parquet / sklearn helpers).


In [ ]:
# 1. Install standard data science packages and core PyTorch silently
!pip install pandas numpy scikit-learn pyarrow matplotlib torch -q

# 2. Install PyTorch Geometric
!pip install torch-geometric -q

import torch
import torch_geometric

print(
    "Dependencies installed |",
    f"torch={torch.__version__} |",
    f"torch_geometric={torch_geometric.__version__} |",
    f"cuda={torch.cuda.is_available()}",
)

## 1. Environment and configuration

Set up the shared runtime for the rest of the notebook:

- Imports (`numpy`, `pandas`, `pathlib`, etc.) used by cleaning and later modelling cells.
- A single `STGAEConfig` dataclass that holds **all** parameters in one place: Drive paths, train/eval years, rolling window length, model size, learning rate, and normative-mask thresholds.
- Fixed random seeds so tensor construction / training runs are more reproducible across Colab sessions.

**Google Drive layout used**

| Path | Role |
|---|---|
| `/content/drive/MyDrive/Dissertation/` | Project root on Drive |
| `.../route_time_panel_v1.parquet` | Joined analysis panel (input) |
| `.../artifacts/st_gae/` | Saved cleaning QA, scaler, model checkpoints (output) |


**Key config fields (defaults)**

| Field | Default | Meaning |
|---|---|---|
| `train_years` | `(2016,)` | Years used to learn healthy flow |
| `eval_years` | `(2017,)` | Held-out year for anomaly scoring / validation |
| `window_size` | `12` | 12 × 5 min = **60 min** temporal context for the GRU |
| `delay_percentile` | `0.85` | Per-route delay cutoff for the normative mask |
| `precip_threshold_mm` | `2.5` | Heavy-precip cutoff excluded from training baseline |
| `feature_cols` / `scale_cols` | travel, delay, weather, cyclical time, calendar flags | Model inputs; only continuous cols are scaled with **train-only** stats later |


In [ ]:
from __future__ import annotations

import json
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd

# Google Drive project root as seen from a Colab runtime (/content/drive/...).
DRIVE_ROOT = Path("/content/drive/MyDrive/Dissertation")


@dataclass(frozen=True)
class STGAEConfig:
    # --- Paths ---
    panel_path: Path = DRIVE_ROOT / "route_time_panel_v1.parquet"
    artifact_dir: Path = DRIVE_ROOT / "artifacts" / "st_gae"

    # --- Temporal train / eval protocol ---
    train_years: tuple[int, ...] = (2016,)  # Calendar years used to learn healthy (normative) traffic dynamics.
    eval_years: tuple[int, ...] = (2017,)   # Held-out year(s) for anomaly scoring and collision-based validation.

    # --- Model / training hyperparameters ---
    window_size: int = 12   # Rolling history length in 5-min steps (12 → 60 minutes of context for the GRU).
    batch_size: int = 32    # Number of window samples per optimizer step.
    hidden_dim: int = 64    # Width of GCN / GRU latent state.
    lr: float = 1e-3        # Adam learning rate for reconstruction training.
    epochs: int = 30        # Full passes over the training window dataset.
    seed: int = 42          # Seed for numpy / python RNGs (and torch later) for reproducibility.

    # --- Normative / cleaning thresholds (healthy-flow baseline) ---
    # Delay quantile computed *inside* each context stratum (see delay_context_keys) 
    # so expected rush-hour delay stays "normal" and can be learned by the ST-GAE.
    delay_percentile: float = 0.85

    # Columns that define a delay-context stratum. Keep order stable for QA/debug.
    # - route_id: corridor-specific delay scale
    # - hour: time-of-day (0–23); captures AM/PM peaks
    # - dow: day-of-week (Mon=0 … Sun=6); captures Friday vs Sunday patterns
    delay_context_keys: tuple[str, ...] = ("route_id", "hour", "dow")

    # If a stratum has fewer samples than this, fall back to coarser keys
    # (drop dow → route+hour, then route only) so sparse cells stay stable.
    delay_context_min_count: int = 30
    
    # Precipitation (mm/hour) above this is treated as non-normal weather for training.
    # Kept separate from the delay quantile so rain is an exclusion flag, not mixed into
    # the definition of "typical Rush hour delay".
    precip_threshold_mm: float = 2.5

    # Minimum Bluetooth sample_count required to treat a bin as observed.
    min_sample_count: int = 0

    # --- Feature schema ---
    # Continuous columns scaled with train-only mean/std (no eval leakage).
    scale_cols: tuple[str, ...] = (
        "travel_time_s",  # corridor travel time (seconds)
        "delay_s",  # travel_time_s - free_flow_s
        "wx_temp_c",  # hourly temperature (°C)
        "wx_precip_mm",  # hourly precipitation (mm)
    )

    # Full model input vector at each corridor × time (after cleaning / encoding).
    feature_cols: tuple[str, ...] = (
        "travel_time_s",  # scaled continuous
        "delay_s",  # scaled continuous
        "wx_temp_c",  # scaled continuous
        "wx_precip_mm",  # scaled continuous
        "hour_sin",  # cyclical hour-of-day
        "hour_cos",  # cyclical hour-of-day
        "dow_sin",  # cyclical day-of-week
        "dow_cos",  # cyclical day-of-week
        "is_public_holiday",  # calendar flag (0/1)
        "is_weekend",  # calendar flag (0/1)
    )


# Instantiate
CFG = STGAEConfig()

# Make cleaning / sampling deterministic across re-runs of this notebook.
random.seed(CFG.seed)
np.random.seed(CFG.seed)

print("DRIVE_ROOT:", DRIVE_ROOT)
print("panel_path:", CFG.panel_path)
print("artifact_dir:", CFG.artifact_dir)
print("config:", json.dumps({k: str(v) for k, v in asdict(CFG).items()}, indent=2))

# Prefer GPU when Colab provides one; fall back to CPU otherwise.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Mount Google Drive and load the joined analysis panel

Mount Google Drive in Colab, then read `route_time_panel_v1.parquet` from `MyDrive/Dissertation/`.

In [ ]:
from google.colab import drive

# 1. Mount Google Drive (Colab maps it under /content/drive)
drive.mount("/content/drive")

# 2. Load dataset from the Dissertation folder on Drive
file_path = CFG.panel_path

# 3. Create artifacts directory
CFG.artifact_dir.mkdir(parents=True, exist_ok=True)

raw = pd.read_parquet(file_path)
raw["ts_local"] = pd.to_datetime(raw["ts_local"], utc=False)

print("loaded:", file_path)
print("rows:", f"{len(raw):,}")
print("years:", sorted(raw["year"].unique().tolist()))
print("routes:", raw["route_id"].nunique())
print("columns:", list(raw.columns))
display(raw.head(3))
display(raw.isna().mean().sort_values(ascending=False).head(12).to_frame("null_rate"))

## 3. Data cleaning/preprocessing

Deduplicate, enforce dtypes, rebuild calendar encodings, create an **observation mask** so missing weather / travel times are not silently treated as zeros after scaling.

In [ ]:
BOOL_COLS = [
    "is_public_holiday",
    "is_civic_holiday",
    "is_holiday",
    "is_school_break",
    "is_mega_event",
    "is_parade",
    "is_shopping_peak",
    "is_weekend",
]


def clean_panel(df: pd.DataFrame, *, min_sample_count: int) -> pd.DataFrame:
    """Return cleaned_df: modelling-ready panel with observation flags and cyclical time features."""
    cleaned_df = df.copy()

    # One row per corridor × timestamp (panel joins can leave rare dupes).
    before = len(cleaned_df)
    cleaned_df = cleaned_df.drop_duplicates(subset=["ts_local", "route_id"], keep="last")
    n_dupes = before - len(cleaned_df)

    cleaned_df = cleaned_df.sort_values(["ts_local", "route_id"]).reset_index(drop=True)

    # Re-derive calendar fields from local timestamps (DST-safe source of truth).
    cleaned_df["hour"] = cleaned_df["ts_local"].dt.hour.astype("int16")
    cleaned_df["dow"] = cleaned_df["ts_local"].dt.dayofweek.astype("int16")
    cleaned_df["is_weekend"] = cleaned_df["dow"].isin([5, 6])

    cleaned_df["hour_sin"] = np.sin(2 * np.pi * cleaned_df["hour"] / 24.0)
    cleaned_df["hour_cos"] = np.cos(2 * np.pi * cleaned_df["hour"] / 24.0)
    cleaned_df["dow_sin"] = np.sin(2 * np.pi * cleaned_df["dow"] / 7.0)
    cleaned_df["dow_cos"] = np.cos(2 * np.pi * cleaned_df["dow"] / 7.0)

    for col in BOOL_COLS:
        if col in cleaned_df.columns:
            cleaned_df[col] = cleaned_df[col].fillna(False).astype(bool)

    # Observation quality: usable Bluetooth measurement?
    cleaned_df["obs_travel"] = (
        cleaned_df["travel_time_s"].notna()
        & cleaned_df["delay_s"].notna()
        & (cleaned_df["sample_count"].fillna(0) >= min_sample_count)
    )
    cleaned_df["obs_weather"] = cleaned_df["wx_temp_c"].notna()  # precip may be sparse; temp is denser

    # Keep precip NaNs as missing rather than inventing dry weather.
    cleaned_df["wx_precip_mm"] = pd.to_numeric(cleaned_df["wx_precip_mm"], errors="coerce")

    cleaned_df.attrs["n_dupes_dropped"] = int(n_dupes)
    return cleaned_df


panel = clean_panel(raw, min_sample_count=CFG.min_sample_count)
print(
    f"cleaned rows={len(panel):,} | dupes_dropped={panel.attrs['n_dupes_dropped']:,} | "
    f"obs_travel={panel['obs_travel'].mean():.1%} | obs_weather={panel['obs_weather'].mean():.1%}"
)


## 4. Normative training mask (healthy-flow baseline)

Flag corridor-times that represent *ordinary* operating conditions for ST-GAE training.

The outoencoder learns healthy dynamics excluding:
- route-matched collisions (`n_collisions > 0`)
- civic / mega disruptions and road-closing events
- heavy precipitation
- **context-extreme** delay (above the delay percentile *within* the same route × hour × day-of-week)
- unobserved / low-sample bins

**Context-aware delay** A flat per-route percentile would treat Friday 17:00 like Sunday 03:00. Instead we compare each observation to the typical delay distribution for that corridor at that time-of-week. Expected Friday-evening congestion can therefore remain in the training baseline; only unusually bad delays *for that context* are excluded. Sparse strata fall back to coarser keys (`route+hour`, then `route`).

Weather is handled separately via `precip_threshold_mm` (exclusion flag), not mixed into the delay quantile.

Held-out years and inference keep *both* normal and abnormal intervals so reconstruction error can surface risk.

In [ ]:
def context_delay_threshold(
    df: pd.DataFrame,
    *,
    delay_percentile: float,
    context_keys: tuple[str, ...],
    min_count: int,
) -> pd.Series:
    """Per-row delay cutoff using stratified quantiles with coarse fallbacks.

    Primary keys default to (route_id, hour, dow) so Friday evenings are judged
    against other Friday evenings on the same corridor—not against overnight lows.
    """
    thr = pd.Series(np.nan, index=df.index, dtype="float64")
    # Try finest stratum first, then drop trailing keys one at a time.
    for width in range(len(context_keys), 0, -1):
        keys = list(context_keys[:width])
        missing = [k for k in keys if k not in df.columns]
        if missing:
            raise KeyError(f"delay context keys missing from panel: {missing}")

        counts = df.groupby(keys, sort=False)["delay_s"].transform("count")
        quantiles = df.groupby(keys, sort=False)["delay_s"].transform(
            lambda s: s.quantile(delay_percentile)
        )
        fillable = thr.isna() & (counts >= min_count) & quantiles.notna()
        thr = thr.where(~fillable, quantiles)

    # Final fallback: any remaining NaN uses the global percentile on observed delays.
    if thr.isna().any():
        global_thr = float(df["delay_s"].quantile(delay_percentile))
        thr = thr.fillna(global_thr)
    return thr


def compute_normal_mask(
    df: pd.DataFrame,
    *,
    delay_percentile: float,
    precip_threshold_mm: float,
    delay_context_keys: tuple[str, ...] = ("route_id", "hour", "dow"),
    delay_context_min_count: int = 30,
) -> pd.Series:
    """Binary mask: 1 = eligible for normative training loss, 0 = excluded."""
    delay_thr = context_delay_threshold(
        df,
        delay_percentile=delay_percentile,
        context_keys=delay_context_keys,
        min_count=delay_context_min_count,
    )

    no_collision = df["n_collisions"].fillna(0).eq(0)
    no_event_pressure = (
        df["n_events_active"].fillna(0).eq(0)
        & df["n_events_road_close"].fillna(0).eq(0)
        & ~df.get("is_mega_event", False).astype(bool)
        & ~df.get("is_parade", False).astype(bool)
        & ~df.get("is_holiday", False).astype(bool)
    )
    # Missing precip: treat as unknown weather → exclude from *training* baseline
    # (do not pretend it was dry).
    precip = df["wx_precip_mm"]
    normal_weather = precip.notna() & (precip < precip_threshold_mm)
    # Delay is "normal" if it is not extreme *for this corridor/time-of-week context*.
    normal_flow = df["delay_s"].notna() & (df["delay_s"] <= delay_thr)
    observed = df["obs_travel"] & df["obs_weather"]

    return (
        observed & no_collision & no_event_pressure & normal_weather & normal_flow
    ).astype("float32")


panel["is_normal"] = compute_normal_mask(
    panel,
    delay_percentile=CFG.delay_percentile,
    precip_threshold_mm=CFG.precip_threshold_mm,
    delay_context_keys=CFG.delay_context_keys,
    delay_context_min_count=CFG.delay_context_min_count,
)

qa_mask = (
    panel.groupby("year")
    .agg(
        n_rows=("route_id", "size"),
        normal_rate=("is_normal", "mean"),
        collision_rate=("n_collisions", lambda s: float((s.fillna(0) > 0).mean())),
        precip_missing=("wx_precip_mm", lambda s: float(s.isna().mean())),
    )
    .reset_index()
)
display(qa_mask)
print(f"Overall normal baseline rate: {panel['is_normal'].mean():.2%}")
print(
    "delay context:",
    CFG.delay_context_keys,
    "| min_count:",
    CFG.delay_context_min_count,
)


In [ ]:
## 5. Train / eval year split + persist cleaning QA

Slice the cleaned panel into train years and eval years, and write a QA JSON next to model artifacts.

In [ ]:
train_df = panel.loc[panel["year"].isin(CFG.train_years)].copy()
eval_df = panel.loc[panel["year"].isin(CFG.eval_years)].copy()

if train_df.empty or eval_df.empty:
    raise ValueError(
        f"Empty train/eval split. train_years={CFG.train_years}, eval_years={CFG.eval_years}"
    )

cleaning_qa = {
    "panel_path": str(CFG.panel_path),
    "n_rows_all": int(len(panel)),
    "n_routes": int(panel["route_id"].nunique()),
    "train_years": list(CFG.train_years),
    "eval_years": list(CFG.eval_years),
    "n_rows_train": int(len(train_df)),
    "n_rows_eval": int(len(eval_df)),
    "normal_rate_all": float(panel["is_normal"].mean()),
    "normal_rate_train": float(train_df["is_normal"].mean()),
    "normal_rate_eval": float(eval_df["is_normal"].mean()),
    "feature_cols": list(CFG.feature_cols),
    "scale_cols": list(CFG.scale_cols),
    "window_size": CFG.window_size,
    "delay_percentile": CFG.delay_percentile,
    "delay_context_keys": list(CFG.delay_context_keys),
    "delay_context_min_count": CFG.delay_context_min_count,
    "precip_threshold_mm": CFG.precip_threshold_mm,
}
qa_path = CFG.artifact_dir / "cleaning_qa.json"
qa_path.write_text(json.dumps(cleaning_qa, indent=2), encoding="utf-8")

print(f"train: {len(train_df):,} rows | normal={train_df['is_normal'].mean():.2%}")
print(f"eval : {len(eval_df):,} rows | normal={eval_df['is_normal'].mean():.2%}")
print("wrote", qa_path)


## 6. Feature tensorization ($T \times N \times D$)
Convert the tidy corridor×time panel into a dense 3D tensor for the ST-GAE:


In [ ]:
from sklearn.preprocessing import StandardScaler

BOOL_FEATURE_COLS = [c for c in CFG.feature_cols if c.startswith("is_")]
CYCLICAL_COLS = [c for c in CFG.feature_cols if c.endswith("_sin") or c.endswith("_cos")]


def panel_to_grid(df: pd.DataFrame, route_ids: list[str], feature_cols: tuple[str, ...]) -> tuple[pd.DataFrame, pd.DatetimeIndex]:
    """Reindex one year-slice onto a complete (ts_local × route_id) grid."""
    frame = df.drop_duplicates(subset=["ts_local", "route_id"], keep="last").copy()
    time_index = pd.DatetimeIndex(sorted(frame["ts_local"].unique()), name="ts_local")
    grid = pd.MultiIndex.from_product([time_index, route_ids], names=["ts_local", "route_id"])
    gridded = (
        frame.set_index(["ts_local", "route_id"])
        .reindex(grid)
        .reset_index()
    )
    # Missing cells after reindex: not observed / not normal for training loss.
    gridded["is_normal"] = gridded["is_normal"].fillna(0.0).astype("float32")
    gridded["obs_travel"] = gridded["obs_travel"].fillna(False).astype(bool)
    gridded["obs_weather"] = gridded["obs_weather"].fillna(False).astype(bool)
    return gridded, time_index


def fit_scaler_on_train_normal(train_grid: pd.DataFrame, scale_cols: tuple[str, ...]) -> StandardScaler:
    """Fit scaler only on healthy, observed train rows (no eval leakage)."""
    fit_mask = (
        train_grid["is_normal"].eq(1.0)
        & train_grid["obs_travel"]
        & train_grid["obs_weather"]
        & train_grid[list(scale_cols)].notna().all(axis=1)
    )
    fit_rows = train_grid.loc[fit_mask, list(scale_cols)]
    if fit_rows.empty:
        raise ValueError("No rows available to fit StandardScaler (check normative mask).")
    scaler = StandardScaler()
    scaler.fit(fit_rows.to_numpy(dtype=np.float64))
    print(f"scaler fit on {len(fit_rows):,} train-normal rows | cols={list(scale_cols)}")
    return scaler


def apply_scaler_and_stack(
    grid: pd.DataFrame,
    *,
    route_ids: list[str],
    feature_cols: tuple[str, ...],
    scale_cols: tuple[str, ...],
    scaler: StandardScaler,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return X (T,N,D), normal_mask (T,N), obs_mask (T,N)."""
    frame = grid.copy()
    # Scale continuous cols; leave cyclical / boolean flags unscaled.
    scaled = frame[list(scale_cols)].to_numpy(dtype=np.float64)
    # sklearn rejects NaNs: temporarily fill for transform, then restore obs mask.
    nan_mask = np.isnan(scaled)
    scaled_filled = np.where(nan_mask, 0.0, scaled)
    scaled_out = scaler.transform(scaled_filled)
    scaled_out = np.where(nan_mask, 0.0, scaled_out)
    frame.loc[:, list(scale_cols)] = scaled_out

    for col in BOOL_FEATURE_COLS:
        if col in frame.columns:
            frame[col] = frame[col].fillna(False).astype(np.float32)
    for col in CYCLICAL_COLS:
        if col in frame.columns:
            frame[col] = frame[col].fillna(0.0).astype(np.float32)

    T = frame["ts_local"].nunique()
    N = len(route_ids)
    D = len(feature_cols)
    # Ensure route order is stable and complete.
    frame["route_id"] = pd.Categorical(frame["route_id"], categories=route_ids, ordered=True)
    frame = frame.sort_values(["ts_local", "route_id"])

    X = frame[list(feature_cols)].to_numpy(dtype=np.float32).reshape(T, N, D)
    normal_mask = frame["is_normal"].to_numpy(dtype=np.float32).reshape(T, N)
    obs_mask = (
        frame["obs_travel"].to_numpy(dtype=bool) & frame["obs_weather"].to_numpy(dtype=bool)
    ).astype(np.float32).reshape(T, N)
    # Training loss should only see normal ∩ observed.
    train_loss_mask = (normal_mask * obs_mask).astype(np.float32)
    return X, train_loss_mask, obs_mask


# Shared corridor inventory from the full cleaned panel (stable node order).
route_ids = sorted(panel["route_id"].astype(str).unique().tolist())
N = len(route_ids)
print(f"corridors N={N}")

train_grid, train_times = panel_to_grid(train_df, route_ids, CFG.feature_cols)
eval_grid, eval_times = panel_to_grid(eval_df, route_ids, CFG.feature_cols)

scaler = fit_scaler_on_train_normal(train_grid, CFG.scale_cols)
X_train, mask_train, obs_train = apply_scaler_and_stack(
    train_grid,
    route_ids=route_ids,
    feature_cols=CFG.feature_cols,
    scale_cols=CFG.scale_cols,
    scaler=scaler,
)
X_eval, mask_eval, obs_eval = apply_scaler_and_stack(
    eval_grid,
    route_ids=route_ids,
    feature_cols=CFG.feature_cols,
    scale_cols=CFG.scale_cols,
    scaler=scaler,
)

print("X_train", X_train.shape, "| mask_train", mask_train.shape, f"| normal∩obs={mask_train.mean():.2%}")
print("X_eval ", X_eval.shape, "| mask_eval ", mask_eval.shape, f"| normal∩obs={mask_eval.mean():.2%}")

# Persist transforms for realtime / Colab restarts.
import pickle

artifact_meta = {
    "route_ids": route_ids,
    "feature_cols": list(CFG.feature_cols),
    "scale_cols": list(CFG.scale_cols),
    "train_years": list(CFG.train_years),
    "eval_years": list(CFG.eval_years),
    "window_size": CFG.window_size,
}
(CFG.artifact_dir / "tensor_meta.json").write_text(
    json.dumps(artifact_meta, indent=2), encoding="utf-8"
)
with (CFG.artifact_dir / "scaler.pkl").open("wb") as handle:
    pickle.dump(scaler, handle)
print("wrote", CFG.artifact_dir / "scaler.pkl")
print("wrote", CFG.artifact_dir / "tensor_meta.json")


## 7. Physical corridor graph (adjacency $A$)

Build a directed graph over **corridors** (`route_id` nodes). An edge $u \rightarrow v$ exists when corridor $u$’s destination detector matches corridor $v$’s origin detector (head–tail connectivity from `SOURCE_TARGET` IDs).

In [ ]:
def split_route_id(route_id: str) -> tuple[str, str]:
    """Parse Bluetooth corridor id 'SOURCE_TARGET' into detector endpoints."""
    parts = str(route_id).split("_")
    if len(parts) != 2 or not parts[0] or not parts[1]:
        raise ValueError(f"Expected SOURCE_TARGET route_id, got {route_id!r}")
    return parts[0], parts[1]


def build_corridor_edge_index(route_ids: list[str]) -> torch.Tensor:
    """Return edge_index shape (2, E) for PyG from head–tail detector matching."""
    endpoints = {rid: split_route_id(rid) for rid in route_ids}
    # Map detector -> corridor indices that *start* at that detector.
    starts_at: dict[str, list[int]] = {}
    for idx, rid in enumerate(route_ids):
        src, _tgt = endpoints[rid]
        starts_at.setdefault(src, []).append(idx)

    sources: list[int] = []
    targets: list[int] = []
    for i, rid in enumerate(route_ids):
        _src, tgt = endpoints[rid]
        for j in starts_at.get(tgt, []):
            # Directed succession: finish corridor i, then start corridor j.
            if i == j:
                continue
            sources.append(i)
            targets.append(j)

    if not sources:
        raise ValueError(
            "No head–tail corridor links found; check route_id naming (SOURCE_TARGET)."
        )

    edge_index = torch.tensor([sources, targets], dtype=torch.long)
    return edge_index


edge_index = build_corridor_edge_index(route_ids).to(device)
print(
    f"edge_index {tuple(edge_index.shape)} | "
    f"E={edge_index.shape[1]} | denser-than-line-ok for branching corridors"
)
# Quick structural QA: degree summary on CPU
deg = torch.bincount(edge_index[0].cpu(), minlength=N).float()
print(
    f"out-degree mean={deg.mean():.2f} | "
    f"median={deg.median():.0f} | max={deg.max():.0f} | isolates={(deg == 0).sum().item()}"
)

torch.save(edge_index.cpu(), CFG.artifact_dir / "edge_index.pt")
print("wrote", CFG.artifact_dir / "edge_index.pt")
